# Diseño de recompensa normalizado: explorador interactivo

Este notebook deja de ser una lámina estática y pasa a ser una herramienta de exploración. Cada figura principal tiene sliders para mover los parámetros que realmente cambian el criterio de decisión.

La lógica vive en `reward_design_interactive_methods.py`. Ese archivo mantiene una estructura parecida al sistema real: diccionario de configuración con forma de `config_CartPole`, un orquestador y módulos internos para normalización, recompensa principal, coordinación y composición. No se modifica `config/config_CartPole.yaml`.


## 0. Carga de la arquitectura de exploración

La configuración inicial preserva la forma de `reward_base -> reward_config -> reward_composition` y `reward_base -> reward_calculation -> metric_processing/principal_reward/coordination_reward/...`. Esto permite que luego agreguemos `extra_rewards`, `local_reward_extension` u otros módulos sin cambiar la manera de explorar.


In [2]:
from copy import deepcopy
from pathlib import Path
import pprint
import sys

import numpy as np

CANDIDATE_DIRS = [
    Path.cwd(),
    Path.cwd() / "Diseño de Recompensa",
]
NOTEBOOK_DIR = next(
    (path for path in CANDIDATE_DIRS if (path / "reward_design_interactive_methods.py").exists()),
    Path.cwd(),
)
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

from reward_design_interactive_methods import (
    build_initial_reward_design_config,
    RewardDesignOrchestrator,
    RewardDesignPlotter,
)

REWARD_DESIGN_CONFIG = build_initial_reward_design_config()
orchestrator = RewardDesignOrchestrator(REWARD_DESIGN_CONFIG)
plotter = RewardDesignPlotter(orchestrator)

print("Bloques activos de reward_calculation:")
pprint.pp(REWARD_DESIGN_CONFIG["reward_base"]["reward_calculation"].keys())


Bloques activos de reward_calculation:
dict_keys(['metric_processing', 'principal_reward', 'coordination_reward', 'local_reward_extension', 'extra_rewards'])


In [3]:
try:
    import ipywidgets as widgets
    from IPython.display import display
    HAS_WIDGETS = True
except Exception as exc:
    HAS_WIDGETS = False
    widgets = None
    print("ipywidgets no está disponible en este kernel. Las funciones plotter.* se pueden llamar manualmente.")
    print(exc)

import matplotlib.pyplot as plt

def show_config_branch(*keys):
    branch = REWARD_DESIGN_CONFIG
    for key in keys:
        branch = branch[key]
    pprint.pp(branch, sort_dicts=False)


## 1. Normalización de `L_e` y `L_edot`

**Pregunta que responde el gráfico.** ¿Los rangos declarados convierten el error y su derivada en costos comparables sin borrar la banda de estabilización?

Mueve los rangos para observar cuándo la señal se vuelve demasiado permisiva o demasiado saturada. La banda verde viene de los criterios físicos de estabilización llevados al espacio normalizado.


In [4]:
if HAS_WIDGETS:
    display(widgets.interactive(
        plotter.plot_normalization_ranges,
        pendulum_e_span=widgets.FloatSlider(value=0.35, min=0.05, max=0.80, step=0.01, description="p_e span", continuous_update=False),
        pendulum_edot_span=widgets.FloatSlider(value=1.20, min=0.10, max=2.50, step=0.05, description="p_edot", continuous_update=False),
        cart_e_span=widgets.FloatSlider(value=0.30, min=0.05, max=0.80, step=0.01, description="c_e span", continuous_update=False),
        cart_edot_span=widgets.FloatSlider(value=0.25, min=0.05, max=1.00, step=0.01, description="c_edot", continuous_update=False),
    ))
else:
    plotter.plot_normalization_ranges()


interactive(children=(FloatSlider(value=0.35, continuous_update=False, description='p_e span', max=0.8, min=0.…

## 2. Recompensa principal: superficie de costo local `J_v`

**Pregunta que responde el gráfico.** ¿La combinación `L_e`/`L_edot` penaliza bien estar lejos, oscilar cerca del objetivo y usar esfuerzo solo cuando corresponde?

El mapa usa la misma forma que `nonlinear_local_cost`:  
`E_v = alpha_e L_e + (1-alpha_e) L_edot`  
`J_v = (E_v + beta_u exp(-kappa E_v) L_u) / (1 + beta_u)`.


In [5]:
if HAS_WIDGETS:
    display(widgets.interactive(
        plotter.plot_local_cost_surface,
        alpha_e=widgets.FloatSlider(value=0.65, min=0.05, max=0.95, step=0.05, description="alpha_e", continuous_update=False),
        effort_weight=widgets.FloatSlider(value=0.10, min=0.00, max=0.60, step=0.02, description="beta_u", continuous_update=False),
        effort_gate_kappa=widgets.FloatSlider(value=4.0, min=0.0, max=10.0, step=0.5, description="kappa", continuous_update=False),
        L_u_fixed=widgets.FloatSlider(value=0.10, min=0.0, max=1.0, step=0.05, description="L_u fijo", continuous_update=False),
        var_obj=widgets.Dropdown(options=["pendulum_angle", "cart_position"], value="pendulum_angle", description="var"),
    ))
else:
    plotter.plot_local_cost_surface()


interactive(children=(FloatSlider(value=0.65, continuous_update=False, description='alpha_e', max=0.95, min=0.…

## 3. Regularización de esfuerzo `L_u`

**Pregunta que responde el gráfico.** ¿`L_u` actúa como regularizador cerca del objetivo o está frenando la recuperación?

La curva muestra la compuerta `g_u(E_v)` y el peso efectivo que realmente multiplica a `L_u`.


In [ ]:
if HAS_WIDGETS:
    display(widgets.interactive(
        plotter.plot_effort_gate,
        alpha_e=widgets.FloatSlider(value=0.65, min=0.05, max=0.95, step=0.05, description="alpha_e", continuous_update=False),
        effort_weight=widgets.FloatSlider(value=0.10, min=0.00, max=0.60, step=0.02, description="beta_u", continuous_update=False),
        effort_gate_kappa=widgets.FloatSlider(value=4.0, min=0.0, max=10.0, step=0.5, description="kappa", continuous_update=False),
        var_obj=widgets.Dropdown(options=["pendulum_angle", "cart_position"], value="pendulum_angle", description="var"),
    ))
else:
    plotter.plot_effort_gate()


interactive(children=(FloatSlider(value=0.65, continuous_update=False, description='alpha_e', max=0.95, min=0.…

## 4. Coordinación global y composición local/global

**Pregunta que responde el gráfico.** ¿El bonus coordinativo incentiva progreso global sin tapar un costo local alto?

El segundo panel permite ver el punto de quiebre: con composición `principal/coordination`, el bonus máximo `C=1` solo compensa hasta `J_v = coordination_weight / principal_weight`.


In [ ]:
if HAS_WIDGETS:
    display(widgets.interactive(
        plotter.plot_progress_and_composition,
        principal_weight=widgets.FloatSlider(value=0.75, min=0.10, max=1.00, step=0.05, description="w_principal", continuous_update=False),
        coordination_weight=widgets.FloatSlider(value=0.25, min=0.00, max=0.90, step=0.05, description="w_coord", continuous_update=False),
        progress_clip_max=widgets.FloatSlider(value=0.02, min=0.002, max=0.08, step=0.002, description="clip Phi", continuous_update=False),
    ))
else:
    plotter.plot_progress_and_composition()


interactive(children=(FloatSlider(value=0.75, continuous_update=False, description='w_principal', max=1.0, min…

## 5. Reparto de crédito coordinativo

**Pregunta que responde el gráfico.** ¿La contribución correctiva de cada lazo reparte el bonus de forma proporcional y estable?

Este gráfico usa la misma idea del módulo coordinativo: `share_v = c_v / (epsilon + sum(c))`, donde `c_v` representa contribución correctiva positiva promedio.


In [ ]:
if HAS_WIDGETS:
    display(widgets.interactive(
        plotter.plot_credit_share,
        epsilon_credit=widgets.FloatLogSlider(value=1.0e-6, base=10, min=-9, max=-2, step=1, description="epsilon", continuous_update=False),
    ))
else:
    plotter.plot_credit_share()


interactive(children=(FloatLogSlider(value=1e-06, continuous_update=False, description='epsilon', max=-2.0, mi…

## 6. Configuración inicial de propuesta

Este diccionario es la propuesta en la misma estructura conceptual de tu configuración. La idea es que cada slider modifique esta estructura internamente mediante el orquestador, y que después podamos agregar más vistas manteniendo el mismo camino de acceso.


In [ ]:
show_config_branch("reward_base", "reward_config", "reward_composition")
show_config_branch("reward_base", "reward_calculation", "principal_reward", "nonlinear_local_cost_params")
show_config_branch("reward_base", "reward_calculation", "coordination_reward")
